# Dört bariyer · eğitim → doğrulama → kilit → test

2/5, 1/1, 1/2 ve 1.5/2 ATR; 10 saat ufuk; aynı mumda çift temas: zarar önce.
Altı temel özellik, GRU, BCE, wavelet kapalı. Üç seed, toplam 12 eğitim.
Bu yeni tek dönem çalışmasıdır; eski 38-fold sonuçlarıyla birleştirilmez.

| Bölüm | UTC dönem | Görev |
|---|---|---|
| Eğitim | 18 Aralık 2025–15 Temmuz 2026 | Ağırlık ve scaler öğrenimi |
| Doğrulama | 17 Temmuz–8 Ağustos 2026 | En düşük BCE ile epoch seçimi |
| Eşik doğrulaması | 10–30 Ağustos 2026 | Ortalama F1 ile ortak aday eşiği |
| Test | 2–14 Ekim 2026 | Kilitli 4 aday, 312 saat; seçim yapılmaz |

Ağustos geçmişte görülmüştür; burada geliştirme verisidir. Eylül test edilmez.
Test verisi yalnızca tüm modeller/eşikler kilitlendikten sonra indirilir.
Kısa tek dönem testi kârlılık veya istatistiksel kesinlik kanıtı değildir.
Farklı bariyerlerin etiketleri farklıdır; ham F1/AP/BCE tek başına kazanan belirlemez.

**Colab Tesla T4** seçin. Drive: **icinbedava34@gmail.com** hesabını kontrol edin.
Eski arşiv salt okunur. Bu notebook kendi yeni workspace'ini kullanır.
Önce prepare → train → freeze → bundle; calibration raporunu inceleyin.
Test için ayrı hücreyi, eşik/model değiştirmeden çalıştırın. Kesintide aynı ayarlarla devam edin.


## 1 — Çalışma ayarları

Aynı deneye devam ederken WORKSPACE_ID ve sürümü değiştirmeyin.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

MODE = 'research'  # recovery: yalnızca tablolar; research: yeni deney; archive: finali oku
PHASE = 'basic'
WORKSPACE_ID = 'barriers_october_v1_001'
EXPERIMENTS = ['B25', 'B11', 'B12', 'B152']  # Her aday ayrı çalıştırılabilir; freeze tümünü ister.
ALLOW_TRAINING = False
RELEASE_TAG = 'advisor-barriers-v1.0.0'  # Sabit yayın; dalın güncel ucunu otomatik çekmez.
REPO_URL = 'https://github.com/umutergul74/YeniBotDeney.git'
REPO_DIR = Path('/content/advisor_barriers_repo')
RUNTIME_DIR = Path('/content/advisor_barriers_runtime')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot/advisor_experiments')
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
assert WORKSPACE_ID.replace('_','').replace('-','').isalnum()
assert MODE in ('recovery','research','archive')
assert PHASE != 'final' or MODE == 'archive', 'Görülmüş final testi yeniden çalıştırılmaz.'

ALLOW_TEST = False


## 2 — Drive ve ayrı çıktı alanı

Hazırlanmış tablolar, ağırlıklar ve ilerleme kalıcı alana yazılır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
LEGACY_ROOT = DRIVE_BASE / 'colab_v1'
namespace = {'recovery':'recovery_v2','research':'barrier_research_v1','archive':'archive_audit_v2'}[MODE]
ROOT = DRIVE_BASE / namespace / WORKSPACE_ID
ROOT.mkdir(parents=True,exist_ok=True)
LOG_DIR = ROOT / 'sessions' / SESSION
LOG_DIR.mkdir(parents=True,exist_ok=True)
print('Eski arşiv (salt okunur):',LEGACY_ROOT)
print('Bu çalışmanın kalıcı alanı:',ROOT)
print('Oturum günlükleri:',LOG_DIR)


## 3 — Sabit Git yayını

Yerel değişiklik varsa otomatik silinmez; yanlış sürümle devam edilmez.


In [ ]:
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',RELEASE_TAG,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
def git(*args):
    return subprocess.check_output(['git','-C',str(REPO_DIR),*args],text=True).strip()
assert not git('status','--porcelain','--untracked-files=no'), 'Kodda yerel değişiklik var.'
assert git('rev-parse','HEAD') == git('rev-parse',RELEASE_TAG+'^{commit}'), 'Yanlış yayın; yeni runtime açın.'
print('Kod sürümü:',git('rev-parse','HEAD'))

assert git('remote','get-url','origin') == REPO_URL, 'Yanlış depo'


## 4 — Kilitli araştırma ortamı

Sistem Python'una pip install yapılmaz. Ayrı Python + venv kullanılır; bütün
bağımlılıklar sürüm ve SHA ile sabittir. Drive'daki indirme önbelleği tekrar kullanılır.
uv kilitleri ve paket onbellegi /content altinda tutulur. Drive kalici ciktilari
ve dogrulanmis kurucu dosyasini saklar. Yeni runtime paketleri yeniden indirebilir.
Kurulum başarısızsa sonraki hücreye geçmeyin. Eğitim için Tesla T4 seçin.


In [ ]:
def logged(command, name):
    with (LOG_DIR/name).open('a',encoding='utf-8',buffering=1) as log:
        env=dict(os.environ)
        for key in ('PYTHONPATH','PYTHONHOME','PYTHONSTARTUP','IPYTHONDIR','JUPYTER_CONFIG_DIR'):
            env.pop(key,None)
        env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
        env['MPLBACKEND']='Agg'  # Colab inline backend is host-only.
        with subprocess.Popen(command,cwd=REPO_DIR,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line,end=''); log.write(line)
                code=process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
            if code: raise subprocess.CalledProcessError(code,command)
RECEIPT=RUNTIME_DIR/'environment_receipt.json'
logged([sys.executable,'-I',str(REPO_DIR/'scripts/bootstrap_advisor.py'),
        '--repo',str(REPO_DIR),'--runtime',str(RUNTIME_DIR),
        '--cache',str(DRIVE_BASE/'reproducibility_cache_v2'),
        '--receipt',str(RECEIPT)],'environment_setup.log')
runtime=json.loads(RECEIPT.read_text())
RESEARCH_PYTHON=runtime['python']
print('İzole Python:',RESEARCH_PYTHON)
print('Paket kilidi:',runtime['lock_sha256'])


def execute_step(action):
    command = [RESEARCH_PYTHON, '-I', str(REPO_DIR/'scripts/advisor_barrier_training.py'), action,
               '--root', str(ROOT), '--legacy', str(LEGACY_ROOT), '--session', SESSION,
               '--candidates', *EXPERIMENTS]
    if ALLOW_TRAINING: command.append('--allow-training')
    if ALLOW_TEST: command.append('--allow-test')
    logged(command, action+'.log')


## 5 · Hazırlık

Mevcut fit_raw dosyasının SHA kontrolü; Ekim verisi okunmaz. Eksik arşivde durur.


In [ ]:
execute_step('prepare')


## 6 · Eğitim

ALLOW_TRAINING=True yapın. Her epoch checkpoint Drive’da. Aynı workspace’e ikinci oturum yazdırmayın.


In [ ]:
if ALLOW_TRAINING:
    execute_step('train')
else:
    print('Eğitim kapalı; ayarlarda ALLOW_TRAINING=True seçin.')


## 7 · Eşik seçimi ve değişmez model kilidi

12 eğitim tamamlanmalıdır. Eşik: 0–1, adım 0.005; üç seed ortalama F1. Eşitlikte yüksek eşik. Aday sıralaması: kalibrasyon BCE skill; testten önce kaydedilir.


In [ ]:
if ALLOW_TRAINING:
    execute_step('freeze')
else:
    print('Eğitim ve kilitleme henüz başlatılmadı.')


## 8 · Rapor ve yedek

Küçük ZIP modelleri içermez. Tam yedek için workspace’in tamamını koruyun.


In [ ]:
execute_step('bundle')
print('Raporlar:', ROOT/'reports')
print('Modeller:', ROOT/'runs')


## 9 · Nihai test — ayrı adım

Önce calibration_REPORT.md ve calibration_thresholds.png dosyalarını inceleyin. ALLOW_TEST=True olduğunda yalnızca kilitli adaylar değerlendirilir. Sonuç kötü olsa da bu testte yeniden ayar yapılmaz.


In [ ]:
if ALLOW_TEST:
    execute_step('test')
    execute_step('bundle')
else:
    print('Test kapalı. Önce eğitim ve kilit; sonra yalnızca bu hücreyi çalıştırın.')
